# 02 - Prétraitement des Données

Nettoyage complet des 79 colonnes pour Lyon, Paris, Bordeaux — juin 2025.

**4 étapes :**
- **3a.** Sélection des 26 colonnes utiles sur 79
- **3b.** Suppression des doublons (sur `id`)
- **3c.** Gestion des valeurs manquantes
- **3d.** Conversion des types (`"$120.00"` → float, `"t"/"f"` → 1/0, `"95%"` → 95.0)

In [ ]:
import sys, os, re
import pandas as pd
import numpy as np
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

CITIES = {'Lyon': 'lyon', 'Paris': 'paris', 'Bordeaux': 'bordeaux'}
raws = {n: pd.read_csv(f'../data/raw/{k}/listings_detail.csv', low_memory=False)
        for n, k in CITIES.items()}
print({n: df.shape for n, df in raws.items()})

### 3a. Sélection des colonnes — de 79 à 26

On garde uniquement les colonnes utiles pour prédire le prix.

| Colonne | Rôle |
|---------|------|
| `price` | Variable cible |
| `accommodates` | Capacité max (très corrélée au prix) |
| `bedrooms`, `beds`, `bathrooms_text` | Taille du logement |
| `room_type`, `property_type` | Catégorie du bien |
| `review_scores_*` | Qualité perçue |
| `host_is_superhost` | Statut hôte premium |

`[c for c in SELECTED if c in df.columns]` : ne garde que les colonnes qui existent réellement.

In [ ]:
SELECTED = [
    'id', 'neighbourhood_cleansed', 'room_type', 'property_type',
    'accommodates', 'bedrooms', 'beds', 'bathrooms_text',
    'price', 'minimum_nights', 'maximum_nights', 'availability_365',
    'number_of_reviews', 'number_of_reviews_ltm', 'reviews_per_month',
    'review_scores_rating', 'review_scores_cleanliness',
    'review_scores_location', 'review_scores_value',
    'host_is_superhost', 'host_response_rate', 'host_acceptance_rate',
    'calculated_host_listings_count', 'instant_bookable',
    'latitude', 'longitude',
]

dfs = {}
for name, df in raws.items():
    cols = [c for c in SELECTED if c in df.columns]
    dfs[name] = df[cols].copy()
    print(f'{name:10s} : {dfs[name].shape}')

### 3b. Suppression des doublons

`duplicated(subset=['id']).sum()` : compte les lignes avec un `id` déjà vu.
`drop_duplicates(subset=['id'])` : garde seulement la première occurrence.

In [ ]:
for name, df in dfs.items():
    n_dup = df.duplicated(subset=['id']).sum()
    print(f'{name:10s} : {n_dup} doublon(s) détecté(s)')
    if n_dup > 0:
        dfs[name] = df.drop_duplicates(subset=['id']).reset_index(drop=True)

### 3c. Gestion des valeurs manquantes

**Colonnes fréquemment à NaN :**
- `reviews_per_month` : NaN si jamais noté → on met **0**
- `review_scores_*` : NaN si pas encore noté → on met **0**
- `bedrooms`, `beds` : parfois non renseigné → on met **1** (valeur la plus courante)
- `price` NaN → **supprimé** (impossible d'entraîner sans variable cible)

In [ ]:
print('Valeurs manquantes avant traitement :')
for name, df in dfs.items():
    missing = df.isnull().sum()
    print(f'\n{name} :')
    print(missing[missing > 0].to_string())

In [ ]:
FILL = {
    'reviews_per_month': 0.0, 'number_of_reviews': 0,
    'number_of_reviews_ltm': 0, 'review_scores_rating': 0.0,
    'review_scores_cleanliness': 0.0, 'review_scores_location': 0.0,
    'review_scores_value': 0.0, 'bedrooms': 1.0, 'beds': 1.0,
}

for name, df in dfs.items():
    for col, val in FILL.items():
        if col in df.columns:
            df[col] = df[col].fillna(val)
    # Supprime les lignes sans prix : impossible d'entraîner sans variable cible
    dfs[name] = df.dropna(subset=['price']).reset_index(drop=True)

print('Après traitement :')
for name, df in dfs.items():
    print(f'  {name:10s} : {df.shape}')

### 3d. Conversion des types

| Colonne | Avant | Après | Méthode |
|---------|-------|-------|---------|
| `price` | `"$120.00"` | `120.0` | str.replace + pd.to_numeric |
| `bathrooms_text` | `"1.5 baths"` | `1.5` | regex re.findall |
| `host_is_superhost` | `"t"/"f"` | `1/0` | dict.map |
| `host_response_rate` | `"95%"` | `95.0` | str.replace + pd.to_numeric |

`errors='coerce'` : les valeurs non convertibles deviennent NaN (au lieu de planter).

In [ ]:
def extract_bathrooms(text):
    """Extrait le nombre depuis '1.5 baths' -> 1.5"""
    if pd.isna(text): return 1.0
    nums = re.findall(r'[\d.]+', str(text))
    return float(nums[0]) if nums else 1.0

for name, df in dfs.items():
    # Prix : supprimer $ et , puis convertir
    df['price'] = (df['price'].astype(str)
                   .str.replace(r'[$,]', '', regex=True)
                   .pipe(pd.to_numeric, errors='coerce'))
    # Salles de bain : extraire le nombre du texte
    if 'bathrooms_text' in df.columns:
        df['bathrooms'] = df['bathrooms_text'].apply(extract_bathrooms)
        df.drop(columns=['bathrooms_text'], inplace=True)
    # Pourcentages : enlever % et convertir
    for col in ['host_response_rate', 'host_acceptance_rate']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace('%','',regex=False),
                                    errors='coerce').fillna(0)
    # Booléens AirBnB : 't' = 1, 'f' = 0
    for col in ['host_is_superhost', 'instant_bookable']:
        if col in df.columns:
            df[col] = df[col].map({'t':1,'f':0,True:1,False:0}).fillna(0).astype(int)
    # Entiers
    for col in ['minimum_nights','maximum_nights','number_of_reviews',
                'number_of_reviews_ltm','calculated_host_listings_count',
                'availability_365','accommodates']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    dfs[name] = df[df['price'] > 0].reset_index(drop=True)

print('Types après conversion (Lyon) :')
print(dfs['Lyon'].dtypes)
for name, df in dfs.items():
    print(f'{name:10s} : {df.shape}')

## Sauvegarde

`index=False` : n'écrit pas l'index pandas dans le CSV (évite la colonne `Unnamed: 0` au rechargement).

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
for name, key in CITIES.items():
    dfs[name].to_csv(f'../data/processed/{key}_clean.csv', index=False)
    print(f'{name} -> {dfs[name].shape}')